**Part 2 — NumPy & Pandas Data Wrangling**
This notebook analyses local YouTube trending CSV files using NumPy
and Pandas. Country files and categories.csv are stored in data/.
No Part 1 files or external data downloads are required.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files
folder = Path("/content/part2_numpy_pandas")
(folder / "data").mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    (folder / "data" / Path(name).name).write_bytes(content)

Saving categories.csv to categories.csv
Saving GBvideos.csv to GBvideos.csv
Saving generate_dataset.py to generate_dataset.py
Saving INvideos.csv to INvideos.csv
Saving README.md to README.md
Saving USvideos.csv to USvideos.csv


**1. NumPy Arrays**

Load the four count columns into NumPy arrays and report their
shape and dtype. Use sorting and slicing to select view counts.

In [ ]:
df = pd.concat([pd.read_csv(p) for p in (folder / "data").glob("*videos.csv")], ignore_index=True)
views, likes, dislikes, comments = df[["views", "likes", "dislikes", "comment_count"]].to_numpy().T
for name, a in zip(["views", "likes", "dislikes", "comments"], [views, likes, dislikes, comments]):
    print(name, a.shape, a.dtype)
print("Top 10:", np.sort(views)[-10:][::-1])
print("Every 100th:", views[::100])
print("Sub-array:", views[10:20])

views (46230,) float64
likes (46230,) float64
dislikes (46230,) float64
comments (46230,) float64
Top 10: [3.60760164e+08 3.43297002e+08 2.86714357e+08 2.82812154e+08
 2.55362869e+08 2.36275528e+08 2.30704137e+08 2.10747294e+08
 2.00444300e+08 1.81785943e+08]
Every 100th: [2.17915000e+05 1.01188300e+06 2.66983000e+05 2.35006000e+05
 3.27020000e+04 3.08573000e+05 8.86960000e+04 2.07630000e+05
 4.66640000e+05 1.42797000e+05 1.86034600e+06 1.67962000e+05
 3.73995300e+06 9.48640000e+04 1.90337000e+05 3.54622000e+05
 2.19825000e+05 7.87420000e+04 3.70580000e+05 3.65804000e+05
 5.31600000e+03 4.91914000e+05 3.60488100e+06 4.03698000e+05
 2.83749100e+06 1.72258000e+05 1.14546000e+05 6.38176000e+05
 9.67373000e+05 1.96627000e+05 2.84969000e+05 3.23012000e+05
 8.61940000e+04 3.66882000e+05 8.04490000e+04 5.67039000e+05
 9.01800000e+04 4.26327000e+05 1.78804000e+05 8.73442000e+05
 2.37628000e+05 4.28204000e+05 9.07338000e+05 5.72380000e+04
 1.90630000e+04 7.52919000e+05 7.16800000e+04 2.55416000

**2. Engagement score without loops**

Weights: 20% views, 50% likes and 30% comments.
Likes indicate positive responses, comments show participation,
and views represent reach. Zero-denominator ratios use zero.
Log scaling reduces the large differences between view counts.
The top 10% flag includes ties at the 90th percentile.

In [ ]:
like_ratio = likes / np.where(likes + dislikes == 0, 1, likes + dislikes)
comment_rate = comments / np.where(views == 0, 1, views)

def normalize(a):
    difference = a.max() - a.min()
    return (a - a.min()) / np.where(difference == 0, 1, difference)

score = 0.2 * normalize(views) + 0.5 * normalize(likes) + 0.3 * normalize(comments)
log_views = np.log1p(views)
top_mask = score >= np.percentile(score, 90)
display(pd.DataFrame({"Like ratio": like_ratio, "Comment rate": comment_rate, "Score": score,
                      "Log views": log_views, "Top 10%": top_mask}).head())

,Like ratio,Comment rate,Score,Log views,Top 10%
0,0.916283,0.003038,NaN,12.291865,False
1,0.905155,0.001817,NaN,11.652818,False
2,0.847249,0.007153,NaN,10.840149,False
3,0.944836,0.003983,NaN,11.873191,False
4,0.940909,0.004977,NaN,12.874469,False


In [ ]:
data = pd.concat([pd.read_csv(p).assign(country=p.stem[:2])
                  for p in (folder / "data").glob("*videos.csv")])
data["day"] = pd.to_datetime(data["trending_date"], format="%y.%d.%m")
daily = data.groupby(["day", "country"]).size().unstack()
display(daily.head())

country,GB,IN,US
day,,,
2017-11-14,200.0,200.0,200.0
2017-11-15,200.0,201.0,202.0
2017-11-16,200.0,200.0,200.0
2017-11-17,200.0,201.0,201.0
2017-11-18,201.0,203.0,200.0


**3. Reshape and Broadcasting**

Rows represent days; columns represent countries.
flatten() converts the matrix into one long array.
reshape() restores its original shape.
Broadcasting calculates country shares and compares daily counts
with each country's average. Missing observations remain NaN.

In [ ]:
counts = daily.to_numpy(dtype=float)
flat = counts.flatten()
reshaped = flat.reshape(counts.shape)
day_total = np.nansum(counts, axis=1, keepdims=True)
country_average = np.nanmean(counts, axis=0, keepdims=True)
share = counts / day_total
relative = counts / country_average
print("Country shares:", share[:3])
print("Compared with country average:", relative[:3])

Country shares: [[0.33333333 0.33333333 0.33333333]
 [0.33167496 0.33333333 0.33499171]
 [0.33333333 0.33333333 0.33333333]]
Compared with country average: [[0.99502488 0.99502488 0.99502488]
 [0.99502488 1.         1.00497512]
 [0.99502488 0.99502488 0.99502488]]


**4. Loading and Inspection**

Load country files, inspect each dataset, correct data types
and identify data-quality issues.

In [ ]:
#1.Load Files:

frames = []
for file in (folder / "data").glob("*videos.csv"):
    table = pd.read_csv(file)
    table["country"] = file.stem[:2]
    frames.append(table)

In [ ]:
#2. Inspect each country and combined data:

df = pd.concat(frames, ignore_index=True)
for table in frames + [df]:
    print(table["country"].unique(), table.shape)
    display(table.head(), table.describe(include="all"))
    table.info()

['IN'] (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,571.0,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,1942.0,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
count,15075,15075,15075,15075,15075.000000,15075,15075,1.507500e+04,1.507100e+04,15075.000000,15063.000000,15075,15075,15075,15075,14483,15075
unique,5878,75,4990,408,NaN,5874,2833,NaN,NaN,NaN,NaN,5878,2,2,2,18,1
top,aGeu6FBmeRM,17.18.11,Rook & Vale - Festival (Live),OfficialReview41,NaN,2017-11-16T09:34:58.000Z,[none],NaN,NaN,NaN,NaN,https://i.ytimg.com/vi/aGeu6FBmeRM/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
freq,23,203,31,659,NaN,23,1209,NaN,NaN,NaN,NaN,23,14730,14903,15045,884,15075
mean,NaN,NaN,NaN,NaN,19.836086,NaN,NaN,9.545924e+05,3.321833e+04,1937.757877,3210.599283,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,7.341271,NaN,NaN,3.035647e+06,1.042103e+05,5791.482855,8251.638153,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,3.542000e+03,0.000000e+00,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,17.000000,NaN,NaN,1.521655e+05,4.107000e+03,257.000000,496.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,24.000000,NaN,NaN,3.558020e+05,1.091500e+04,681.000000,1243.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,24.000000,NaN,NaN,8.639980e+05,2.926200e+04,1815.500000,3101.000000,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                15075 non-null  object 
 1   trending_date           15075 non-null  object 
 2   title                   15075 non-null  object 
 3   channel_title           15075 non-null  object 
 4   category_id             15075 non-null  int64  
 5   publish_time            15075 non-null  object 
 6   tags                    15075 non-null  object 
 7   views                   15075 non-null  int64  
 8   likes                   15071 non-null  float64
 9   dislikes                15075 non-null  int64  
 10  comment_count           15063 non-null  float64
 11  thumbnail_link          15075 non-null  object 
 12  comments_disabled       15075 non-null  bool   
 13  ratings_disabled        15075 non-null  bool   
 14  video_error_or_removed  15075 non-null

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
count,16080,16080,16080,16080,16080.000000,16080,16080,1.608000e+04,1.607600e+04,16080.000000,16068.000000,16080,16080,16080,16080,15516,16080
unique,5068,80,4403,408,NaN,5064,2501,NaN,NaN,NaN,NaN,5068,2,2,2,18,1
top,BsTMdCTYs7P,17.30.12,Sunny Park - Bangalore (Live),QuietChallenge75,NaN,2018-01-01T10:45:28.000Z,[none],NaN,NaN,NaN,NaN,https://i.ytimg.com/vi/BsTMdCTYs7P/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
freq,25,205,33,644,NaN,25,1164,NaN,NaN,NaN,NaN,25,15751,15911,16048,960,16080
mean,NaN,NaN,NaN,NaN,18.975871,NaN,NaN,1.012289e+06,3.950464e+04,2115.772015,3118.737989,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,7.590729,NaN,NaN,3.746761e+06,1.719414e+05,7973.927750,10060.176081,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,1.621000e+03,0.000000e+00,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,10.000000,NaN,NaN,1.173078e+05,3.353000e+03,194.000000,375.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,23.000000,NaN,NaN,2.890725e+05,9.229000e+03,540.000000,999.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,24.000000,NaN,NaN,7.507190e+05,2.626700e+04,1506.000000,2613.250000,NaN,NaN,NaN,NaN,NaN,NaN


... (3 outputs left out)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                15075 non-null  object 
 1   trending_date           15075 non-null  object 
 2   title                   15075 non-null  object 
 3   channel_title           15075 non-null  object 
 4   category_id             15075 non-null  int64  
 5   publish_time            15075 non-null  object 
 6   tags                    15075 non-null  object 
 7   views                   15075 non-null  int64  
 8   likes                   15071 non-null  float64
 9   dislikes                15075 non-null  int64  
 10  comment_count           15063 non-null  float64
 11  thumbnail_link          15075 non-null  object 
 12  comments_disabled       15075 non-null  bool   
 13  ratings_disabled        15075 non-null  bool   
 14  video_error_or_removed  15075 non-null

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,571.0,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,1942.0,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
count,46230,46230,46230,46230,46230.000000,46230,46230,4.623000e+04,4.621800e+04,46230.000000,46194.000000,46230,46230,46230,46230,44606,46230
unique,14699,80,10404,438,NaN,14678,6056,NaN,NaN,NaN,NaN,14699,2,2,2,18,3
top,iAvRHGeUpBR,17.30.12,Sunny Park - Bangalore (Live),QuietChallenge75,NaN,2017-11-27T11:50:56.000Z,[none],NaN,NaN,NaN,NaN,https://i.ytimg.com/vi/iAvRHGeUpBR/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
freq,28,608,66,2464,NaN,28,3785,NaN,NaN,NaN,NaN,28,45182,45686,46138,2612,16080
mean,NaN,NaN,NaN,NaN,18.645339,NaN,NaN,1.178603e+06,4.580005e+04,2325.983625,3449.773347,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,7.646745,NaN,NaN,5.916188e+06,2.969554e+05,10518.962756,13665.153495,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,1.066000e+03,0.000000e+00,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,10.000000,NaN,NaN,1.294632e+05,3.656000e+03,213.000000,404.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,22.000000,NaN,NaN,3.182775e+05,1.020100e+04,594.000000,1074.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,24.000000,NaN,NaN,8.240505e+05,2.898775e+04,1666.000000,2779.000000,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  int64  
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        46230 non-null  bool   
 14  video_error_or_removed  46230 non-null

In [ ]:
#3. df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m", errors="coerce", utc=True)

df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce", utc=True)
columns = ["views", "likes", "dislikes", "comment_count", "category_id"]
df[columns] = df[columns].apply(pd.to_numeric, errors="coerce")

In [ ]:
df["trending_date"] = pd.to_datetime(
    df["trending_date"], format="%y.%d.%m", errors="coerce", utc=True
)

In [ ]:
#4. Check issues:

print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated(["country", "video_id", "trending_date"]).sum())
print("Removed videos:\n", df["video_error_or_removed"].value_counts())
print("Negative counts:\n", (df[columns[:4]] < 0).sum())
print("Invalid date order:", (df["trending_date"] < df["publish_time"].dt.normalize()).sum())

Missing values:
 video_id                     0
trending_date                0
title                        0
channel_title                0
category_id                  0
publish_time                 0
tags                         0
views                        0
likes                       12
dislikes                     0
comment_count               36
thumbnail_link               0
comments_disabled            0
ratings_disabled             0
video_error_or_removed       0
description               1624
country                      0
dtype: int64
Duplicates: 230
Removed videos:
 video_error_or_removed
False    46138
True        92
Name: count, dtype: int64
Negative counts:
 views            0
likes            0
dislikes         0
comment_count    0
dtype: int64
Invalid date order: 0


**5. Filtering**

Find records with high views and likes, the top 20 engagement
scores, and Music or Gaming records above the median views.

& requires both conditions; | accepts either condition.
loc selects matching records and columns.
iloc selects the first 20 positions after sorting.
In the original dataset, category 10 is Music and 20 is Gaming.


In [ ]:
#1.Prepare score and readable columns:

df["engagement_score"] = score
columns = ["title", "country", "views", "likes", "engagement_score"]

In [ ]:
#2. 1 million+ views AND more than 50,000 likes:

selected = df.loc[
    (df["views"] >= 1000000) & (df["likes"] > 50000),
    columns
]
display(selected.sort_values("views", ascending=False))

,title,country,views,likes,engagement_score
41601,Lena Ortiz - Monsoon (Live),GB,360760164,27537822.0,NaN
46029,The Hollow Pines - Phones (Official Video) WATCH,GB,343297002,9581910.0,NaN
41400,Lena Ortiz - Monsoon (Live),GB,286714357,21885701.0,NaN
45828,The Hollow Pines - Phones (Official Video) WATCH,GB,282812154,7893691.0,NaN
39183,Rook & Vale - Phones (Official Video) WATCH,GB,255362869,11413793.0,NaN
...,...,...,...,...,...
6699,Brand New Bangalore Mystery,IN,1001786,54433.0,NaN
44312,DJ Aster - Cricket (Live) OFFICIAL!,GB,1001246,118660.0,NaN
41824,NOVA KANE - PASTA (LYRIC VIDEO)?!,GB,1000250,55738.0,NaN
4134,Perfect Paris Speech HD,IN,1000116,56313.0,NaN


In [ ]:
#3. Top 20 engagement scores:

top = df.sort_values("engagement_score", ascending=False)
display(top[columns].iloc[:20])

,title,country,views,likes,engagement_score
0,Sunny Park - Cats (Lyric Video)!,IN,217915,14918.0,NaN
1,Perfect Pasta Unboxing?!,IN,115014,2739.0,NaN
2,Wild Budget Challenge!!,IN,51028,1786.0,NaN
3,Quiet Chess Mystery,IN,143370,5224.0,NaN
4,Quiet Festival Prank,IN,390220,15079.0,NaN
5,Insane Football Review,IN,627323,22445.0,NaN
6,SURPRISING WINTER HIGHLIGHTS LIVE,IN,73747,0.0,NaN
7,Wild Chess Breakdown,IN,550559,11895.0,NaN
8,Surprising Monsoon Tutorial!!,IN,94198,888.0,NaN
9,Mira Solis - Paris (Official Video)?!,IN,866457,103178.0,NaN


In [ ]:
#4. Music OR Gaming above median views:

median = df["views"].median()
category = (df["category_id"] == 10) | (df["category_id"] == 20)
selected = df.loc[category & (df["views"] > median), columns]
display(selected.sort_values("views", ascending=False))

,title,country,views,likes,engagement_score
41601,Lena Ortiz - Monsoon (Live),GB,360760164,27537822.0,NaN
46029,The Hollow Pines - Phones (Official Video) WATCH,GB,343297002,9581910.0,NaN
41400,Lena Ortiz - Monsoon (Live),GB,286714357,21885701.0,NaN
45828,The Hollow Pines - Phones (Official Video) WATCH,GB,282812154,7893691.0,NaN
39183,Rook & Vale - Phones (Official Video) WATCH,GB,255362869,11413793.0,NaN
...,...,...,...,...,...
19784,Sunny Park - Cats (Live),US,318748,16196.0,NaN
40068,Echo District - Guitar (Lyric Video) LEAKED,GB,318742,0.0,NaN
7781,Sunny Park - Space (Lyric Video),IN,318736,12642.0,NaN
40088,DJ Aster - Chess (Live),GB,318720,18251.0,NaN


**6. Missing Values and Duplicates**
Report missing counts and percentages, fill optional text,
drop records missing essential values and remove duplicate records.

In [ ]:
#1. Missing counts and percentages:

missing = pd.DataFrame({
    "Count": df.isna().sum(),
    "Percentage": df.isna().mean() * 100
})
display(missing[missing["Count"] > 0])

,Count,Percentage
likes,12,0.025957
comment_count,36,0.077872
description,1624,3.512870
engagement_score,46230,100.000000


In [ ]:
#2. Fill missing text:

df["description"] = df["description"].fillna("")
df["title"] = df["title"].fillna("Unknown title")
df["channel_title"] = df["channel_title"].fillna("Unknown channel")
df["tags"] = df["tags"].fillna("[none]")

In [ ]:
#3. Drop missing essential values:

required = ["video_id", "country", "trending_date", "publish_time",
            "views", "likes", "dislikes", "comment_count"]
before = len(df)
df = df.dropna(subset=required).copy()
print("Rows removed for missing values:", before - len(df))

Rows removed for missing values: 0


In [ ]:
#4. Remove duplicates:

before = len(df)
df = df.drop_duplicates(subset=["country", "video_id", "trending_date"])
print("Duplicate rows removed:", before - len(df))
print("Remaining rows:", len(df))

**7. Joining Category Names**
Load categories.csv and use a left merge to attach category names.
A left merge keeps all trending records, including unmatched categories.

In [ ]:
#1. Load categories:

categories = pd.read_csv(folder / "data" / "categories.csv")
display(categories.head())

,category_id,category_name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


In [ ]:
#2. Prepare category columns:

categories = categories.rename(columns={"title": "category_name"})
categories = categories[["category_id", "category_name"]].drop_duplicates()
categories["category_id"] = pd.to_numeric(categories["category_id"], errors="raise")
assert categories["category_id"].notna().all()

In [ ]:
#3.Merge and check record counts:

before = len(df)
df = df.merge(categories, on="category_id", how="left", validate="many_to_one")
print("Before merge:", before)
print("After merge:", len(df))
assert len(df) == before, "Record count changed"

Before merge: 46182
After merge: 46182


In [ ]:
#4. Handle unmatched categories:

print("Unmatched records:", df["category_name"].isna().sum())
df["category_name"] = df["category_name"].fillna("Unknown")
display(df[["title", "category_id", "category_name"]].head())

Unmatched records: 0


,title,category_id,category_name
0,Sunny Park - Cats (Lyric Video)!,10,Music
1,Perfect Pasta Unboxing?!,1,Film & Animation
2,Wild Budget Challenge!!,24,Entertainment
3,Quiet Chess Mystery,23,Comedy
4,Quiet Festival Prank,24,Entertainment


**8. Aggregation**
Calculate record counts and average views, likes and engagement
by category and country. Show category counts and percentages,
then compare publishing weekdays and hours.

In [ ]:
#1. Define summary calculations:

metrics = {
    "record_count": ("video_id", "size"),
    "average_views": ("views", "mean"),
    "average_likes": ("likes", "mean"),
    "average_engagement": ("engagement_score", "mean")
}

In [ ]:
#2. Summarise categories and countries:

category_summary = df.groupby("category_name").agg(**metrics)
country_summary = df.groupby("country").agg(**metrics)
display(category_summary)
display(country_summary)

,record_count,average_views,average_likes,average_engagement
category_name,,,,
Autos & Vehicles,1008,2.796418e+05,8701.510913,NaN
Comedy,3269,6.032311e+05,28724.764454,NaN
Education,1651,3.620306e+05,12584.478498,NaN
Entertainment,12324,9.000517e+05,28908.899464,NaN
Film & Animation,2306,5.765017e+05,17199.381613,NaN
Gaming,2072,5.094763e+05,20746.993243,NaN
Howto & Style,1833,3.007170e+05,10256.982542,NaN
Music,10351,3.189727e+06,134848.846392,NaN
News & Politics,2428,2.835561e+05,3108.582784,NaN


,record_count,average_views,average_likes,average_engagement
country,,,,
GB,15059,1.581263e+06,65134.136662,NaN
IN,15059,9.544438e+05,33217.693207,NaN
US,16064,1.012406e+06,39509.171439,NaN


In [ ]:
#3. Category distribution:

distribution = pd.DataFrame({
    "Count": df["category_name"].value_counts(),
    "Percentage": df["category_name"].value_counts(normalize=True) * 100
})
display(distribution)

,Count,Percentage
category_name,,
Entertainment,12324,26.685722
Music,10351,22.413494
People & Blogs,3815,8.260794
Comedy,3269,7.078515
News & Politics,2428,5.257460
Film & Animation,2306,4.993287
Gaming,2072,4.486597
Sports,1928,4.174787
Howto & Style,1833,3.969079


In [ ]:
#4. Prepare publishing times:

videos = df.sort_values("trending_date")
videos = videos.drop_duplicates(["country", "video_id"], keep="last").copy()
videos["weekday"] = videos["publish_time"].dt.day_name()
videos["hour"] = videos["publish_time"].dt.hour

In [ ]:
#5. Compare weekdays:

days = videos.groupby("weekday").agg(
    video_count=("video_id", "size"), average_views=("views", "mean")
)
display(days.sort_values("average_views", ascending=False))
print("Highest-view weekday:", days["average_views"].idxmax())

,video_count,average_views
weekday,,
Monday,2151,1.192618e+06
Friday,2074,1.020937e+06
Tuesday,2291,1.009512e+06
Wednesday,2168,9.325229e+05
Thursday,2125,9.281497e+05
Saturday,2157,8.507699e+05
Sunday,2089,7.852371e+05


Highest-view weekday: Monday


In [ ]:
#6. Compare hours:

hours = videos.groupby("hour").agg(
    video_count=("video_id", "size"), average_views=("views", "mean")
)
display(hours.sort_values("average_views", ascending=False))
print("Highest-view hour (UTC):", hours["average_views"].idxmax())

**9. Time to Trend and Trending Lifespan**
Calculate the calendar days between publication and trending.
For each video within each country, count its distinct trending days.
List the 10 longest-lasting videos and average lifespan by category.

In [ ]:
#1. Calculate days to trend:

published = df["publish_time"].dt.normalize()
df["days_to_trend"] = (df["trending_date"] - published).dt.days
print("Negative delays:", (df["days_to_trend"] < 0).sum())
df.loc[df["days_to_trend"] < 0, "days_to_trend"] = np.nan
display(df["days_to_trend"].describe())

Negative delays: 0


,days_to_trend
count,46182.000000
mean,5.095362
std,6.248737
min,0.000000
25%,1.000000
50%,3.000000
75%,6.000000
max,47.000000


In [ ]:
#2. First observed time to trend:

first_delay = df.groupby(["country", "video_id"])["days_to_trend"].min()
display(first_delay.describe())

,days_to_trend
count,15055.000000
mean,2.804517
std,5.631879
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,38.000000


In [ ]:
#3. Calculate lifespan:

lifespan = df.groupby(["country", "video_id"]).agg(
    trending_lifespan=("trending_date", "nunique")
).reset_index()
display(lifespan.head())

,country,video_id,trending_lifespan
0,GB,--UQXlrk5E0,4
1,GB,--pb5oKkYrH,4
2,GB,-13iTcoyyzU,1
3,GB,-1VjLKhzB3I,1
4,GB,-1aX3Wk1cJx,4


In [ ]:
#4. Add titles and categories:

details = df.sort_values("trending_date")
details = details.drop_duplicates(["country", "video_id"], keep="last")
details = details[["country", "video_id", "title", "category_name"]]
lifespan = lifespan.merge(details, on=["country", "video_id"], validate="one_to_one")

In [ ]:
#5. Longest lifespan and category averages:

longest = lifespan.sort_values("trending_lifespan", ascending=False)
display(longest.head(10))
average = lifespan.groupby("category_name")["trending_lifespan"].mean()
display(average.sort_values(ascending=False))

,country,video_id,trending_lifespan,title,category_name
1846,GB,RdqSDDng84n,25,Lena Ortiz - Monsoon (Live),Music
3858,GB,w1lZB-FX3Sn,25,DJ Aster - Festival (Official Video) WATCH?!,Music
2343,GB,Z1aFIGyR5zv,25,Nova Kane - Robots (Official Video),Music
11903,US,Nc1MUQvmyWO,25,Sunny Park - Cats (Official Video),Music
2668,GB,cz0dTlZWwSk,25,Final Cars Speech WATCH,Film & Animation
2687,GB,dIbJkzpClRM,25,Sunny Park - Robots (Live),Music
1019,GB,EZaG9_2Yi1u,25,KAIRO - FESTIVAL (LIVE),Music
950,GB,DYLzrqG-SrK,25,Official Football Update,Entertainment
1232,GB,Hh9b0BBsne0,25,Kairo - Phones (Lyric Video)!,Music
3212,GB,lxWhyKE6Ifz,25,Echo District - Football (Official Video),Music


,trending_lifespan
category_name,
Music,5.327470
Entertainment,3.538817
Comedy,2.788346
People & Blogs,2.783150
Film & Animation,2.700000
Gaming,2.642674
Education,2.439169
Howto & Style,2.399209
Autos & Vehicles,2.367059


**10. Combining Data**
Use concat() to stack country datasets. Create video and channel
summaries, then merge them to show each channel's trending-video
count and total views.

In [ ]:
#1. Combine country datasets:

combined = pd.concat(frames, ignore_index=True)
print("Combined shape:", combined.shape)
display(combined.head())

Combined shape: (46230, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,571.0,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,1942.0,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


In [ ]:
#2. Create a per-video summary using cleaned data:

video_summary = df.groupby("video_id").agg(
    video_views=("views", "max")
).reset_index()
display(video_summary.head())

,video_id,video_views
0,--UQXlrk5E0,179541
1,--j83Wp5WrQ,68090
2,--pb5oKkYrH,582515
3,-0B4eg2Srpi,396797
4,-0dHuLKRPYS,198202


In [ ]:
#3. Add channel names:

details = df.sort_values("trending_date")
details = details.drop_duplicates("video_id", keep="last")
details = details[["video_id", "channel_title"]]
video_summary = video_summary.merge(details, on="video_id", validate="one_to_one")

In [ ]:
#4. Create a per-channel summary:

channel_summary = video_summary.groupby("channel_title").agg(
    trending_videos=("video_id", "nunique"),
    total_views=("video_views", "sum")
).reset_index()
display(channel_summary.sort_values("total_views", ascending=False))

,channel_title,trending_videos,total_views
290,QuietChallenge75,466,1333310585
432,WildUnboxing39,56,582453017
133,FinalTrailer17,161,492211013
281,PerfectTutorial78,98,477526034
278,PerfectTrailer65,66,452538325
...,...,...,...
110,EpicTutorial71,1,141712
14,AmazingRecipe40,2,125229
100,EpicMystery67,1,115333
424,WildReview24,1,71334


In [ ]:
#5. Merge video and channel summaries:

result = video_summary.merge(
    channel_summary, on="channel_title", how="left", validate="many_to_one"
)
display(result.head(20))

,video_id,video_views,channel_title,trending_videos,total_views
0,--UQXlrk5E0,179541,UltimateUnboxing65,524,298819629
1,--j83Wp5WrQ,68090,AmazingSpeech11,154,117836877
2,--pb5oKkYrH,582515,QuietChallenge75,466,1333310585
3,-0B4eg2Srpi,396797,FinalBreakdown11,164,143515053
4,-0dHuLKRPYS,198202,OfficialUnboxing6,28,11377223
5,-13iTcoyyzU,179389,SurprisingChallenge82,216,63895309
6,-1B196P9MCk,105724,InsaneReaction46,321,298887790
7,-1VjLKhzB3I,451687,UltimateUnboxing65,524,298819629
8,-1aX3Wk1cJx,89824,FinalBreakdown11,164,143515053
9,-1lyybGIUpD,208313,PerfectVlog13,158,94612809
